使用python 3.11.11 和 selenium 4.4.0
首先打开网页

In [1]:
username = "34520242201240"
password = "123456"
captcha_token = "heCqKwPwf9OpeQJO5W1q7jFelHDWzKjsk5jG0VtwTGc"

from selenium import webdriver
from selenium.webdriver.common.by import By
import time
import requests
from typing import Any

d = webdriver.Chrome()

d.get("https://myoj2.vscode.live:20102/")

print("等待进入登录界面")
while(not d.current_url.startswith("https://myoj2.vscode.live:20102/OJ/account_signin.html")):
    pass

time.sleep(10)


def verify(img_base64: str)->str:
    url = "http://api.jfbym.com/api/YmServer/customApi"

    data: dict[str, Any] = {
        "token": captcha_token,
        "type": 50100,
        "image": img_base64,
    }
    _headers = {"Content-Type": "application/json"}
    response = requests.request("POST", url, headers=_headers, json=data).json()
    print(response)
    return str(response["data"]["data"])

time.sleep(2)

image_data = d.find_element(By.ID, "vcode-image").screenshot_as_base64#type:ignore

d.find_element(By.ID, "user_id").send_keys(username)#type:ignore
d.find_element(By.ID, "password").send_keys(password)#type:ignore
image_ans = verify(image_data)
print(f"获取到验证码结果 {image_ans}")
d.find_element(By.ID,"vcode").send_keys(image_ans)#type:ignore
d.find_element(By.ID, "btn-submit").click() #type:ignore

等待进入登录界面
{'msg': '识别成功', 'code': 10000, 'data': {'code': 0, 'data': '1513', 'time': 4.156532287597656, 'externel': 2, 'file_path': 'https://ali-jfb2024.oss-cn-chengdu.aliyuncs.com/jfb_upload/dabiao/2025/12/80af3b884877bfd63fd3f1b64cf36c5b.png', 'order_unique_id': '80af3b884877bfd63fd3f1b64cf36c5b', 'reduce_score': 12, 'unique_code': '80af3b884877bfd63fd3f1b64cf36c5b'}}
获取到验证码结果 1513


接下来可以进入选课界面，然后以下代码可以自动进行答题

In [2]:
base_url = "https://api.jisuai.top/v1"
api_key = "sk-REDACTED-SEE-README"
model = "gemini-flash-latest"

import fake_useragent
from bs4 import BeautifulSoup

fua = fake_useragent.FakeUserAgent()

try:
    name = d.find_elements(By.CLASS_NAME, "dropdown")[1].text.split(" ")[0]  # type:ignore
except:
    name = d.find_elements(By.CLASS_NAME, "dropdown")[0].text.split(" ")[  # type:ignore
        0
    ]
print(f"登录成功，当前用户 {name}")


def duckduckgo_search(query: str) -> str:
    url = f"https://duckduckgo.com/html/?q={query}"
    response = requests.get(
        url,
        headers={"User-Agent": fua.random},
        proxies={"http": "http://127.0.0.1:7890", "https": "http://127.0.0.1:7890"},
    )
    soup = BeautifulSoup(response.text, "html.parser")
    results = soup.getText()
    if results:
        return results.replace("\n", " ")
    return "No results found."


system_prompt: list[Any] = [
    {
        "role": "system",
        "content": """设想你是一个对于计算机知识极为渊博的专家，请根据问题和这个问题的初步搜索结果决定是否要继续搜索并回答问题""",
    },
    {
        "role": "system",
        "content": """回答问题采用 JSON 格式
总共包含两个字段：
1.answer 这个字段是题目的回答，如 A B C D 或者回答文本，题目可能是单选题或者问答题， O 代表不确定
2.reason 这个字段为 str 类型的一段话，表示给出这个答案的原因
一个示例的回答为：
{"answer":"A","reason":"该计算机名为ENIAC（电子数字积分计算机），于 1946 年 2 月在美国宾夕法尼亚大学正式诞生。"}
请保证回答符合这个格式
你回答的问题都是和C语言相关的，请务必保证回答的正确性""",
    },
    {
        "role": "system",
        "content": f"""几个注意事项：
1.你的名字叫{name}
2.如果要求写代码，代码一定要用C语言书写
3.你写的代码一定要像人一点，不要像AI写的，变量名要有意义，不要有任何注释""",
    },
]

functions: list[Any] = [
    {
        "type": "function",
        "function": {
            "name": "duckduckgo_search",
            "description": "当需要获取实时信息、最新数据或不知道的内容时使用，例如当前事件、天气、新闻等",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "搜索的关键词或问题，需要具体明确",
                    }
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_previous_problem",
            "description": "一般如果题目信息不明确一般都需要之前的题目的信息并且回答当前题目时使用",
            "parameters": {
                "type": "object",
                "properties": {
                    "last_pos": {
                        "type": "integer",
                        "description": "上一题的位置索引，比如上一题就是-1，上两题就是-2，以此类推",
                    },
                },
                "required": [],
            },
        },
    },
]

from selenium.webdriver.remote.webelement import WebElement
from openai import OpenAI
from openai.types.chat import ChatCompletionMessageFunctionToolCall
import json
from typing import Callable

client = OpenAI(api_key=api_key, base_url=base_url)

question_store: list[str] = []

def get_previous_problem(last_pos: int = -1) -> str:
    return question_store[last_pos]

def get_result(text: str, check_func: Callable[[dict[str, Any]], None]):
    result = duckduckgo_search(text)
    user_prompt: list[Any] = [
        {
            "role": "user",
            "content": f"""问题: {text}
            问题的直接搜索结果: {result}
            请注意，如果题目信息不明确务必调用工具获得前一题或者两题的题目信息来辅助回答""",
        },
    ]
    messages = system_prompt + user_prompt
    while True:
        ret = client.chat.completions.create(
            model=model, messages=messages, tools=functions, tool_choice="auto"
        )
        response = ret.choices[0].message
        messages.append(response)
        if response.tool_calls:
            tool_calls = response.tool_calls

            # 处理每个工具调用
            for tool_call in tool_calls:
                if isinstance(tool_call, ChatCompletionMessageFunctionToolCall):
                    function_name = tool_call.function.name
                    function_args = json.loads(tool_call.function.arguments)

                    print(f"调用工具: {function_name}，参数: {function_args}")

                    if function_name == "duckduckgo_search":
                        tool_result = duckduckgo_search(function_args["query"])
                    elif function_name == "get_previous_problem":
                        tool_result = get_previous_problem(function_args.get("last_pos", -1))
                    else:
                        tool_result = "未知的工具"
                    messages.append(
                        {
                            "role": "tool",
                            "tool_call_id": tool_call.id,
                            "content": tool_result,
                        }
                    )
                else:
                    messages.append(
                        {
                            "role": "tool",
                            "tool_call_id": tool_call.id,
                            "content": "未知的工具",
                        }
                    )
        else:
            try:
                if response.content:
                    result = json.loads(
                        response.content.replace("```json", "").replace("```", "")
                    )
                    print(f"获取到题目结果 {result}")
                    check_func(result)
                    return result
                raise ValueError("未知的返回，请保证返回为 JSON 格式")
            except Exception as e:
                print(f"返回错误 {e}")
                messages.append({"role": "assistant", "content": f"返回错误: {e}"})


questions: list[WebElement] = d.find_elements(By.CLASS_NAME, "question")  # type:ignore
start_tag = 0

登录成功，当前用户 王子恒


In [3]:
question_store: list[str] = []
for que in questions[start_tag::]:
    question_ele: WebElement = que.find_element(  # type:ignore
        By.CLASS_NAME, "requirement"
    )

    select_eles: list[WebElement] = que.find_elements(  # type:ignore
        By.TAG_NAME, "ul"
    )
    parent = que.find_element(By.XPATH, "..")  # type:ignore
    if parent.text.find("选择题") != -1:
        state = "单选题"

        def check_func(ret: dict[str, Any]):
            if (len(ret["answer"])) != 1:
                raise ValueError("题目为单选题")

    else:
        select_eles: list[WebElement] = que.find_elements(  # type:ignore
            By.TAG_NAME, "input"
        )
        if len(select_eles) > 0:
            state = "填空题"
        else:
            state = "简答题"

        def check_func(ret: dict[str, Any]):
            if not "answer" in ret:
                raise ValueError("回答必须有 answer 字段")
            if ret["answer"] == "0":
                ret["answer"] = "是0"

    text = state + "\n" + question_ele.get_attribute("innerHTML")  # type:ignore
    print(f"开始回答 {text}")
    while True:
        try:
            result = get_result(text, check_func=check_func)
            answer: str = result["answer"]
            if state == "单选题":
                if answer == "O":
                    print("返回为O，尝试结合上一题回答。")
                    result = get_result(
                        f"上一题信息：“{get_previous_problem()}”"
                        + "\n本题如下："
                        + text,
                        check_func=check_func,
                    )
                    answer: str = result["answer"]
                tag = ord(answer.upper()) - 64
                while True:
                    print(f"开始选择 {answer.upper()}")
                    try:
                        tgt_ele: WebElement = que.find_elements(  # type:ignore
                            By.TAG_NAME, "option"
                        )[tag]
                        tgt_ele.location_once_scrolled_into_view  # type:ignore
                        tgt_ele.click()
                        check_ans = que.find_element(  # type:ignore
                            By.TAG_NAME, "select"
                        ).get_attribute(
                            "value"
                        )  # type:ignore
                        if check_ans == answer.upper():
                            break
                    except Exception as e:
                        print(f"选择答案错误 {e}")
                        time.sleep(1)
                        tgt: WebElement = que.find_element(  # type:ignore
                            By.TAG_NAME, "input"
                        )
                        tgt.location_once_scrolled_into_view  # type:ignore
                        tgt.clear()
                        tgt.send_keys(answer)  # type:ignore
                        check_ans = tgt.get_attribute("value")  # type:ignore
                        if check_ans.replace(" ", "").replace(
                            "\n", ""
                        ) == answer.replace(" ", "").replace("\n", ""):
                            break
                    time.sleep(1)
                break
            elif state == "填空题":
                tgt: WebElement = que.find_element(By.TAG_NAME, "input")  # type:ignore
                tgt.location_once_scrolled_into_view  # type:ignore
                tgt.clear()
                tgt.send_keys(answer)  # type:ignore
                check_ans = tgt.get_attribute("value")  # type:ignore
                if check_ans.replace(" ", "").replace("\n", "") == answer.replace(
                    " ", ""
                ).replace("\n", ""):
                    break
            elif state == "简答题":
                tgt: WebElement = que.find_element(  # type:ignore
                    By.TAG_NAME, "textarea"
                )
                tgt.location_once_scrolled_into_view  # type:ignore
                tgt.clear()
                tgt.send_keys(answer)  # type:ignore
                check_ans = tgt.get_attribute("value")  # type:ignore
                if check_ans == answer:
                    break
        except Exception as e:
            print(f"脚本错误 {e}")
    question_store.append(text)
    start_tag += 1
    print(f"已更新 start_tag = {start_tag}")

开始回答 填空题
<p>函数<code>printf()</code>中用到格式符<code>%5s</code>。如果字符串长度大于5，则输出按方式（              ） </p>
<pre><code>A.  从左起输出该字串，右补空格；

B.  按原字符长从左向右全部输出；

C.  右对齐输出该字串,左补空格；

D.  输出错误信息。
</code></pre><div class="answer row"><div class="col-md-12"><input type="text" class="form-control code" name="B[893]" placeholder="请输入您的答案" value=""></div></div>
获取到题目结果 {'answer': 'B', 'reason': '我是王子恒，对于C语言的标准库函数`printf()`，格式符`%Ns`中的`N`指定的是输出的最小字段宽度。如果待输出的字符串长度小于`N`，则会按照对齐方式（默认右对齐，左边补空格）进行填充。但是，如果字符串的实际长度大于指定的最小宽度`N`，`printf()`会忽略这个最小宽度限制，按字符串的实际长度将整个字符串完整地从左向右输出，以确保信息不丢失。因此，如果字符串长度大于5，会按原字符长全部输出。'}
已更新 start_tag = 1
开始回答 填空题
<p>以下叙述不正确的是（           ）</p>
<pre><code>A.  一个C源程序可由一个或多个函数组成；

B.  一个C源程序必须包含一个`main`函数；

C.  C程序的基本组成单位是函数；

D.  在C程序中，注释只能位于一条语句的后面。
</code></pre><div class="answer row"><div class="col-md-12"><input type="text" class="form-control code" name="B[894]" placeholder="请输入您的答案" value=""></div></div>
脚本错误 Error code: 429 - {'error': {'message': '该令牌状态不可用 (request id: 2025122617592091870